# Can we predict `failed_90d` after an inspection?

**Question:** from measurements taken at inspection time, can we predict whether the asset fails within 90 days?

Plan: inspect the data, check for leakage, build a leakage-free baseline and a gradient-boosting model, and evaluate on a **time-based holdout** (train on earlier inspections, test on the latest), since that mirrors real use.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, '| missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum(), '| unique assets:', df.asset_tag.nunique())
print('Failure rate:', df.failed_90d.mean().round(4), '| range:', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## Leakage check: `parts_replaced`
The dictionary says this is the number of parts replaced on the work order *that closed this inspection record*. That work order can be closed after a failure and repair, so it may encode the outcome.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','size'])

Zero parts replaced means 0% failures and 2+ means 100%. That is not a signal available at inspection time; it is the outcome leaking in via the repair record. A model using it would look near-perfect and be useless for prediction. **I exclude `parts_replaced`.** (Worth confirming with whoever owns the work-order system.)

## Failure rate over time

In [ ]:
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean','size'])
q['mean'].plot(kind='bar', title='Failure rate by inspection quarter', ylabel='failed_90d rate'); plt.show()
q

The base rate drifts over time, so a random split could hide problems; I use time-ordered validation. A low rate in the final quarter could also be partly right-censoring (failures not yet observed) — worth checking with the data owner.

## Features vs. outcome

In [ ]:
feats_num = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig, ax = plt.subplots(1,5, figsize=(18,3))
for a,c in zip(ax,feats_num):
    df.groupby(pd.qcut(df[c],10,duplicates='drop'), observed=True).failed_90d.mean().reset_index(drop=True).plot(ax=a, marker='o'); a.set_title(c); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.mean().round(3).to_dict())
print(df[feats_num+['failed_90d']].corr()['failed_90d'].round(3))

## Modeling
Features: the five numeric measurements, `site`, and inspection month (drift/seasonality). Excluded: `parts_replaced` (leakage), `asset_tag` (unique per row), raw timestamp.

Evaluation: train on the first 75% of inspections by time, test on the last 25%; expanding-window CV on the training part. Metrics: ROC AUC and average precision against a prevalence-only baseline.

In [ ]:
df['month'] = df.inspected_at.dt.month
num = feats_num + ['month']; cat = ['site']
X = df[num+cat]; y = df.failed_90d
cut = int(len(df)*0.75)
Xtr, Xte, ytr, yte = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
print('train', len(Xtr), ytr.mean().round(3), '| test', len(Xte), yte.mean().round(3), '| test starts', df.inspected_at.iloc[cut].date())

models = {
 'prevalence baseline': DummyClassifier(strategy='prior'),
 'logistic regression': make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)]), LogisticRegression(max_iter=1000)),
 'gradient boosting': make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)]),
                                    HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
rows=[]; fitted={}
for n,m in models.items():
    cvauc=[]
    for a,b in TimeSeriesSplit(5).split(Xtr):
        m.fit(Xtr.iloc[a], ytr.iloc[a]); cvauc.append(roc_auc_score(ytr.iloc[b], m.predict_proba(Xtr.iloc[b])[:,1]))
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; fitted[n]=(m,p)
    rows.append(dict(model=n, cv_auc=np.mean(cvauc), test_auc=roc_auc_score(yte,p), test_ap=average_precision_score(yte,p), test_brier=brier_score_loss(yte,p)))
pd.DataFrame(rows).set_index('model').round(3)

Bootstrap intervals for holdout AUC, to see how much to trust the numbers:

In [ ]:
rng=np.random.default_rng(0); yt=yte.values
for n in ['logistic regression','gradient boosting']:
    p=fitted[n][1]
    aucs=[roc_auc_score(yt[i],p[i]) for i in (rng.integers(0,len(yt),len(yt)) for _ in range(500))]
    print(n, 'test AUC 95% CI:', np.percentile(aucs,[2.5,97.5]).round(3))

In [ ]:
best = fitted['gradient boosting'][0]
imp = permutation_importance(best, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=0)
pd.Series(imp.importances_mean, index=Xte.columns).sort_values().plot(kind='barh', title='Permutation importance (drop in test AUC)'); plt.show()
lr = fitted['logistic regression'][0]
print(pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out()).round(3).sort_values())

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n in ['logistic regression','gradient boosting']:
    pr,rc,_=precision_recall_curve(yte,fitted[n][1]); ax[0].plot(rc,pr,label=n)
ax[0].axhline(yte.mean(),ls='--',c='gray',label='base rate'); ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].legend(); ax[0].set_title('Precision-recall (time holdout)')
t=pd.DataFrame({'p':fitted['gradient boosting'][1],'y':yte.values}); t['bin']=pd.qcut(t.p,5,duplicates='drop')
g=t.groupby('bin',observed=True).agg(pred=('p','mean'),obs=('y','mean'))
ax[1].plot(g.pred,g.obs,'o-'); m_=g.values.max(); ax[1].plot([0,m_],[0,m_],'--',c='gray'); ax[1].set_xlabel('mean predicted'); ax[1].set_ylabel('observed rate'); ax[1].set_title('Calibration (quintiles)')
plt.tight_layout(); plt.show()
top=t.sort_values('p',ascending=False).head(int(.2*len(t)))
print(f"Top 20% by risk: failure rate {top.y.mean():.3f} vs {t.y.mean():.3f} overall; captures {top.y.sum()/t.y.sum():.0%} of failures")

## Conclusions (numbers from my run of this code; re-run the notebook to regenerate outputs)
- **Yes, moderately.** On a time-based holdout (inspections from 2025-07-12 on, 4.3% failure rate), gradient boosting reaches AUC ≈ 0.76 (95% CI ≈ 0.70–0.81), vs 0.50 for the baseline; logistic regression gets ≈ 0.66. Flagging the top 20% riskiest inspections catches ≈ 52% of failures (11% failure rate vs 4.3% overall), i.e. ~2.5x lift. Precision is still low in absolute terms (average precision ≈ 0.13).
- **Drivers:** higher vibration, load and bearing temperature raise risk; more meter hours and higher ambient temperature lower it in the linear model. `site` carries essentially no signal.
- **`parts_replaced` was excluded as leakage** (0 parts → 0% failures, 2+ → 100%). Using it would look near-perfect but is unavailable at prediction time.
- **Caveats:** the failure rate drifts strongly over time (≈3% to ≈27% by quarter), and the test period is low-rate, so absolute probabilities are not stable; the late-period drop may be right-censoring. Each asset appears once, so no per-asset history features are possible. Confirm with the data owner before deploying.